In [1]:
# load libraries
from atproto import Client
import pandas as pd
from datetime import datetime
import re
import emoji

# load the datasets containing MP data
bundestag_21 = pd.read_csv("../01_data/clean_data/bundestag_21_clean.csv")
bundestag_25 = pd.read_csv("../01_data/clean_data/bundestag_25_clean.csv")

# for each dataset get only the MPs with existing bsky handles
bundestag_21.dropna(subset=["clean_handle"], inplace=True)
bundestag_25.dropna(subset=["clean_handle"], inplace=True)

After having loaded all packages and the data, I drop all rows of politicians that do not have a Bluesky account. Then, I create a client instance to interact with the Bluesky API. To log into the client, I need an app password which I have saved in the same folder, but which is not uploaded to the GitHub repo.

In [4]:
# create a client instance
client = Client()

# get the app password
with open("app_password.txt", "r") as f:
    app_password = f.read()

handle = "mxwlnd.bsky.social"

# login with my credentials
client.login(handle, app_password)

ProfileViewDetailed(did='did:plc:5sqqg66p7muc7ogbp6xx4sw6', handle='mxwlnd.bsky.social', associated=ProfileAssociated(chat=None, feedgens=0, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated', activitySubscription={'allowSubscriptions': 'followers'}), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:5sqqg66p7muc7ogbp6xx4sw6/bafkreigwrjedzb7jvmowkn6fbe2atbnlwecsa4ouk5wpz54eg6rqkvayrq@jpeg', banner=None, created_at='2025-05-19T19:28:35.738Z', description=None, display_name='', followers_count=2, follows_count=1, indexed_at='2025-05-19T19:28:35.738Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=0, verification=None, viewer=ViewerState(blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=None, muted=False, muted_by_list=None, py_type='app.bsky.actor.defs#viewerState'), py_type='app.bsky.actor.defs#profileViewDetailed')

Now I create a function that collects all posts for all Bluesky handles in the politician dataframe that are no reposts and fall in the research period. I need to use a cursor variable to really get all posts and not just the ones that are visible on the first page.

In [5]:
# define a function that cleans the handle for invisible characters and cleans the handle further
def clean_handle(raw_handle):
    return re.sub(r'[\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff]', '', raw_handle.strip().lower())

# define a function that returns a df with all posts from given handle
def retrieve_posts(df):

    # empty list in which all the data is stored
    all_data = []

    # loop over all rows of the MP-level dataframe
    for _, row in df.iterrows():
        # clean the bsky handle and get start and end date of period in which MP served in parliament
        handle = clean_handle(row["clean_handle"])
        faction_start = pd.to_datetime(row["faction_start"])
        faction_end = pd.to_datetime(row["faction_end"])

        # if start date is after research period, this will be the first day to look for posts
        if faction_start >= datetime(2024, 6, 1):
            start_period = faction_start
            # otherwise it will be the start of the research period
        else:
            start_period = datetime(2024, 6, 1)
        
        # correct end date by one day
        if faction_end == datetime(2025, 3, 25):
            end_period = datetime(2025, 3, 24)
            # if field is empty, this means that MP still serves and end date is end of the research period
        elif pd.isnull(row["faction_end"]):
            end_period = datetime(2025, 7, 31)
        else:
            end_period = faction_end

        # get the did for the handle
        try:
            did = client.com.atproto.identity.resolve_handle({'handle': handle})['did']
        except Exception as e:
            print(f"Failed to resolve handle {handle}: {e}")
            continue
        
        # assign cursor and looping variable, needed to get all posts
        cursor = None
        keep_looping = True

        # while Boolean is true, get the whole feed for the did
        while keep_looping == True:
            response = client.app.bsky.feed.get_author_feed({
                'actor': did,
                'cursor': cursor,
                'limit': 100
            })

            # get the response feed
            feed = response['feed']

            # extract the text and date for each post in the feed
            for item in feed:
                post_handle = item["post"]["author"]["handle"]
                text = item["post"]["record"]["text"]
                date = item["post"]["record"]["created_at"]
                date = pd.to_datetime(date).tz_localize(None)

                # filter out reposts and check that post falls in the research period, if true append to list
                if handle == post_handle and start_period <= date <= end_period:
                    post_data = row.to_dict()
                    post_data.update({
                        "text": text,
                        "date": date
                    })
                    all_data.append(post_data)

            # update the cursor, if there is no cursor stop looping
            cursor = response['cursor']
            if not cursor:
                keep_looping = False

    # convert to a df, sort by date and return
    df = pd.DataFrame(all_data).sort_values(by="date", ascending=True)

    return df


I apply the function to the dataframes for both legislatures.

In [6]:
# retrieve the posts for both legislatures
posts_bt_21 = retrieve_posts(bundestag_21)
posts_bt_25 = retrieve_posts(bundestag_25)

Failed to resolve handle achimpost.de: 


Now, I concatenate both dataframes to one comprehensive one. I further clean the text column to remove any user mentions, urls and emojis. I also normalize the whitespaces in between words. Moreover, I remove all rows with empty text fields and rows for posts that contain of less than three words. All these preprocessing steps are done to improve the sentiment classification.

In [7]:
# combine both dfs to one
posts_bt_combined = pd.concat([posts_bt_21, posts_bt_25])

# apply basic text cleaning
def clean_text(text):
    text = re.sub(r"rt\s+@", "", text)           # remove retweet prefix
    text = re.sub(r"@\w+", "", text)             # remove mentions
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"&\w+;", "", text)            # remove HTML entities
    text = emoji.replace_emoji(text, replace="") # remove emojis
    text = re.sub(r'"', "", text)                # remove all quotation marks
    text = re.sub(r"\s+", " ", text).strip()     # normalize whitespace
    return text

# apply the cleaning to all posts
posts_bt_combined["text"] = posts_bt_combined["text"].apply(clean_text)

# remove any NAs
posts_bt_combined = posts_bt_combined.dropna(subset=["text"])

# remove if the text consists of less than 3 words
posts_bt_combined = posts_bt_combined[
    posts_bt_combined["text"].str.strip().str.split().str.len() >= 3
]

# export the dataset as a csv
posts_bt_combined.to_csv("../01_data/clean_data/all_posts_cleaned.csv", index=False)